In [1]:
pip install requests beautifulsoup4 pandas lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import requests
from bs4 import BeautifulSoup

url = "http://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
response.encoding = "utf-8"   # <-- add this line
soup = BeautifulSoup(response.text, "lxml")

soup = BeautifulSoup(response.text, "lxml")

books = soup.find_all("article", class_="product_pod")
print(f"Found {len(books)} books on this page")

# Let's just look at the FIRST book to test our extraction logic
first_book = books[0]

title = first_book.h3.a["title"]
price = first_book.find("p", class_="price_color").text
rating = first_book.p["class"][1]  # class list is ["star-rating", "Three"]
availability = first_book.find("p", class_="instock availability").text.strip()

print("Title:", title)
print("Price:", price)
print("Rating:", rating)
print("Availability:", availability)

Found 20 books on this page
Title: A Light in the Attic
Price: £51.77
Rating: Three
Availability: In stock


In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "http://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
response.encoding = "utf-8"
soup = BeautifulSoup(response.text, "lxml")

books = soup.find_all("article", class_="product_pod")

all_books = []  # this will hold every book's data as a dictionary

for book in books:
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    rating = book.p["class"][1]
    availability = book.find("p", class_="instock availability").text.strip()

    all_books.append({
        "title": title,
        "price": price,
        "rating": rating,
        "availability": availability
    })

df = pd.DataFrame(all_books)
print(df)

                                                title   price rating  \
0                                A Light in the Attic  £51.77  Three   
1                                  Tipping the Velvet  £53.74    One   
2                                          Soumission  £50.10    One   
3                                       Sharp Objects  £47.82   Four   
4               Sapiens: A Brief History of Humankind  £54.23   Five   
5                                     The Requiem Red  £22.65    One   
6   The Dirty Little Secrets of Getting Your Dream...  £33.34   Four   
7   The Coming Woman: A Novel Based on the Life of...  £17.93  Three   
8   The Boys in the Boat: Nine Americans and Their...  £22.60   Four   
9                                     The Black Maria  £52.15    One   
10     Starving Hearts (Triangular Trade Trilogy, #1)  £13.99    Two   
11                              Shakespeare's Sonnets  £20.66   Four   
12                                        Set Me Free  £17.46   

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

all_books = []

for page_num in range(1, 51):  # pages 1 to 50
    url = f"http://books.toscrape.com/catalogue/page-{page_num}.html"
    response = requests.get(url)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "lxml")

    books = soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        rating = book.p["class"][1]
        availability = book.find("p", class_="instock availability").text.strip()

        all_books.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability
        })

    print(f"Scraped page {page_num} — total books so far: {len(all_books)}")
    time.sleep(0.5)  # polite delay so we don't hammer the server

df = pd.DataFrame(all_books)
print(df.shape)
print(df.head())

Scraped page 1 — total books so far: 20
Scraped page 2 — total books so far: 40
Scraped page 3 — total books so far: 60
Scraped page 4 — total books so far: 80
Scraped page 5 — total books so far: 100
Scraped page 6 — total books so far: 120
Scraped page 7 — total books so far: 140
Scraped page 8 — total books so far: 160
Scraped page 9 — total books so far: 180
Scraped page 10 — total books so far: 200
Scraped page 11 — total books so far: 220
Scraped page 12 — total books so far: 240
Scraped page 13 — total books so far: 260
Scraped page 14 — total books so far: 280
Scraped page 15 — total books so far: 300
Scraped page 16 — total books so far: 320
Scraped page 17 — total books so far: 340
Scraped page 18 — total books so far: 360
Scraped page 19 — total books so far: 380
Scraped page 20 — total books so far: 400
Scraped page 21 — total books so far: 420
Scraped page 22 — total books so far: 440
Scraped page 23 — total books so far: 460
Scraped page 24 — total books so far: 480
Scrap

In [8]:
# Clean price: remove £ symbol, convert to float
df["price"] = df["price"].str.replace("£", "").astype(float)

# Convert rating from words to numbers for easier analysis
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["rating"] = df["rating"].map(rating_map)

# Check for duplicates or missing values
print("Duplicates:", df.duplicated().sum())
print("Missing values:\n", df.isnull().sum())

df.drop_duplicates(inplace=True)

# Save the final dataset
df.to_csv("books_dataset.csv", index=False)
print("Saved! Final shape:", df.shape)

Duplicates: 0
Missing values:
 title           0
price           0
rating          0
availability    0
dtype: int64
Saved! Final shape: (1000, 4)


In [9]:
import os
print(os.getcwd())

C:\Users\nisha\Downloads\telco


In [10]:
print(os.path.exists("books_dataset.csv"))
print(os.listdir())

True
['.ipynb_checkpoints', 'books_dataset.csv', 'output_16_1.png', 'output_18_1.png', 'output_18_2.png', 'output_5_2.png', 'output_6_0.png', 'telco data.md', 'Untitled.ipynb', 'Untitled1.ipynb']
